In [12]:
import numpy as np
import matplotlib.pyplot as plt
from tabulate import tabulate


import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [14]:
# model: [depth(d), width(w), max_channels(mc)]
model_scales = {
  "yolo11n": [0.50, 0.25, 1024],
  "yolo11s": [0.50, 0.50, 1024],
  "yolo11m": [0.50, 1.00, 512],
  "yolo11l": [1.00, 1.00, 512],
  "yolo11x": [1.00, 1.50, 512]}

<h3>Helper functions</h3>

In [15]:
def calculate_yolo11_shapes(input_image, model_name, model_configs):

    # Extract model parameters
    if model_name not in model_configs:
        raise ValueError(f"Model configuration for '{model_name}' not found.")

    d, w, mc = model_configs[model_name] # depth, width, max_channels
    height, width, channels = input_image

    h_reduced1 = height // 2; h_reduced2 =  h_reduced1 // 2; h_reduced3 =  h_reduced2 // 2; h_reduced4 =  h_reduced3 // 2; h_reduced5 =  h_reduced4 // 2
    w_reduced1 = width // 2; w_reduced2 =  w_reduced1 // 2; w_reduced3 =  w_reduced2 // 2; w_reduced4 =  w_reduced3 // 2; w_reduced5 =  w_reduced4 // 2
    c_increased1 = int(min(64, mc)*w); c_increased2 = int(min(128, mc)*w); c_increased3 = int(min(256, mc)*w)
    c_increased4 = int(min(512, mc)*w); c_increased5 = int(min(1024, mc)*w)

    table = [
        ["", "Input Image", f"[{height},{width},{channels}]", f"[1,{channels},{height},{width}]"],
        ["0", "Conv Output", f"[{h_reduced1},{w_reduced1},{c_increased1}]", f"[1,{c_increased1},{h_reduced1},{w_reduced1}]"],
        ["1", "Conv Output", f"[{h_reduced2},{w_reduced2},{c_increased2}]", f"[1,{c_increased2},{h_reduced2},{w_reduced2}]"],
        ["2", "C3k2 Output", f"[{h_reduced2},{w_reduced2},{c_increased3}]", f"[1,{c_increased3},{h_reduced2},{w_reduced2}]"],
        ["3", "Conv Output", f"[{h_reduced3},{w_reduced3},{c_increased3}]", f"[1,{c_increased3},{h_reduced3},{w_reduced3}]"],
        ["4", "C3k2 Output", f"[{h_reduced3},{w_reduced3},{c_increased4}]", f"[1,{c_increased4},{h_reduced3},{w_reduced3}]"],
        ["5", "Conv Output", f"[{h_reduced4},{w_reduced4},{c_increased4}]", f"[1,{c_increased4},{h_reduced4},{w_reduced4}]"],
        ["6", "C3k2 Output", f"[{h_reduced4},{w_reduced4},{c_increased4}]", f"[1,{c_increased4},{h_reduced4},{w_reduced4}]"],
        ["7", "Conv Output", f"[{h_reduced5},{w_reduced5},{c_increased5}]", f"[1,{c_increased5},{h_reduced5},{w_reduced5}]"],
        ["8", "C3k2 Output", f"[{h_reduced5},{w_reduced5},{c_increased5}]", f"[1,{c_increased5},{h_reduced5},{w_reduced5}]"],
        ["9", "SPPF Output", f"[{h_reduced5},{w_reduced5},{c_increased5}]", f"[1,{c_increased5},{h_reduced5},{w_reduced5}]"],
        ["10", "C2PSA Output", f"[{h_reduced5},{w_reduced5},{c_increased5}]", f"[1,{c_increased5},{h_reduced5},{w_reduced5}]"],
        ["11", "Upsample Output", f"[{h_reduced4},{w_reduced4},{c_increased5}]", f"[1,{c_increased5},{h_reduced4},{w_reduced4}]"],
        ["12", "Concat Output", f"[{h_reduced4},{w_reduced4},{c_increased4+c_increased5}]", f"[1,{c_increased4+c_increased5},{h_reduced4},{w_reduced4}]"],
        ["13", "C3k2 Output", f"[{h_reduced4},{w_reduced4},{c_increased4}]", f"[1,{c_increased4},{h_reduced4},{w_reduced4}]"],
        ["14", "Upsample Output", f"[{h_reduced3},{w_reduced3},{c_increased4}]", f"[1,{c_increased4},{h_reduced3},{w_reduced3}]"],
        ["15", "Concat Output", f"[{h_reduced3},{w_reduced3},{c_increased4+c_increased4}]", f"[1,{c_increased4+c_increased4},{h_reduced3},{w_reduced3}]"],
        ["16", "C3k2 Output", f"[{h_reduced3},{w_reduced3},{c_increased3}]", f"[1,{c_increased3},{h_reduced3},{w_reduced3}]"],
        ["17", "Conv Output", f"[{h_reduced4},{w_reduced4},{c_increased3}]", f"[1,{c_increased3},{h_reduced4},{w_reduced4}]"],
        ["18", "Concat Output", f"[{h_reduced4},{w_reduced4},{c_increased3+c_increased4}]", f"[1,{c_increased3+c_increased4},{h_reduced4},{w_reduced4}]"],
        ["19", "C3k2 Output", f"[{h_reduced4},{w_reduced4},{c_increased4}]", f"[1,{c_increased4},{h_reduced4},{w_reduced4}]"],
        ["20", "Conv Output", f"[{h_reduced5},{w_reduced5},{c_increased4}]", f"[1,{c_increased4},{h_reduced5},{w_reduced5}]"],
        ["21", "Concat Output", f"[{h_reduced5},{w_reduced5},{c_increased4+c_increased5}]", f"[1,{c_increased4+c_increased5},{h_reduced5},{w_reduced5}]"],
        ["22", "C3k2 Output", f"[{h_reduced5},{w_reduced5},{c_increased5}]", f"[1,{c_increased5},{h_reduced5},{w_reduced5}]"],
        ["23", "Detect Inputs", f"[{h_reduced3},{w_reduced3},{c_increased3}]", f"[1,{c_increased3},{h_reduced3},{w_reduced3}]"],
        ["", "", f"[{h_reduced4},{w_reduced4},{c_increased4}]", f"[1,{c_increased4},{h_reduced4},{w_reduced4}]"],
        ["", "", f"[{h_reduced5},{w_reduced5},{c_increased5}]", f"[1,{c_increased5},{h_reduced5},{w_reduced5}]"]
    ]

    headers = ["Stage", "Block", "Shape [H,W,C]", "Shape [B,C,H,W]"]
    print(tabulate(table, headers=headers, tablefmt="grid"))

In [ ]:
# ANSI escape codes for bold text
BOLD = '\033[1m'
RESET = '\033[0m'

def print_header(text, total_width=80, filler='-'):
    padding = (total_width - len(text)) // 2
    print(filler * padding + text + filler * (total_width - len(text) - padding))

def imshow_tensor(x): # image --> torch.size([1, 3, 640, 640])
    im = x.squeeze(0)
    im = im.permute(1, 2, 0) # [3, 640, 640] --> [640, 640, 3]
    im = im * 255 # [0, 1] --> [0, 255]
    im = im.numpy().astype(np.uint8) # tensor --> numpy array
    plt.imhow(im)
    plt.axis("off")
    plt.show()

In [17]:
input_size = [640, 640, 3]
model_name = "yolo11n"

calculate_yolo11_shapes(input_size, model_name, model_scales)

+---------+-----------------+-----------------+-------------------+
|   Stage | Block           | Shape [H,W,C]   | Shape [B,C,H,W]   |
+=========+=================+=================+===================+
|         | Input Image     | [640,640,3]     | [1,3,640,640]     |
+---------+-----------------+-----------------+-------------------+
|       0 | Conv Output     | [320,320,16]    | [1,16,320,320]    |
+---------+-----------------+-----------------+-------------------+
|       1 | Conv Output     | [160,160,32]    | [1,32,160,160]    |
+---------+-----------------+-----------------+-------------------+
|       2 | C3k2 Output     | [160,160,64]    | [1,64,160,160]    |
+---------+-----------------+-----------------+-------------------+
|       3 | Conv Output     | [80,80,64]      | [1,64,80,80]      |
+---------+-----------------+-----------------+-------------------+
|       4 | C3k2 Output     | [80,80,128]     | [1,128,80,80]     |
+---------+-----------------+-----------------+-

## Conv Block

<h2><strong>Role of Convolutional Layers</strong></h2>

Convolutional layers are the backbone of neural networks for object detection:

- **<u>Feature Extraction:</u>**  
  By applying filters (kernels) to input data, they learn spatial hierarchies of patterns, such as edges, textures, shapes, and objects.

- **<u>Hierarchical Learning:</u>**  
  Stacking multiple layers allows for capturing  complex features, from low-level edges in early layers to high-level representations in deeper layers.

- **<u>Progressive Downsampling:</u>**  
  These layers reduce the spatial dimensions of the input image (e.g., P1 → P2 → P3) while increasing channel depth. This process ensures:  
  - **Efficient Input Reduction:** Minimizes computational costs while retaining critical information.  
  - **Spatial Relationship Preservation:** Maintains the structure and arrangement of data, ensuring meaningful features are retained.

- **<u>Parameter Sharing:</u>**  
  Filters are reused across the input, reducing the number of parameters and enabling translational invariance.


In [18]:
def autopad(k, p = None, d = 1):
    ''' Pad to same shape outputs'''
    if d > 1:
        d = d*(k-1) + 1 if isinstance(k, int) else [d*(x-1) + 1 for x in k] # actual kernel size

    if p is None:
        p = k // 2 if isinstance(k, int) else [x // 2 for x in k] # auto - pad
    return p

class Conv(nn.Module):
    # Standard Convolution 
    default_act = nn.SiLU() # default activation
    
    def __init__(self, c1, c2, k = 1, s = 1, p = None, g = 1, d = 1 , act = True):
        # initialize Conv layer with given parameters
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, k, s, autopad(k, p, d), groups = g, dilation = d, bias = False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = (self.default_act if act is True 
                   else act if isinstance(act, nn.Module) 
                   else nn.Identity()
        )    
    def forward(self, x):
        # forward propagation : convolution --> normalization --> activation to input tensor
        return self.act(self.bn(self.conv(x)))
    
    def forward_fuse(self, x):
        # forward propagation without batch normalization (for inference optimization)
        return self.act(self.conv(x))

## C3K2 (F) Block

<h2><strong>Role of the Bottleneck Block (when c3k = False)</strong></h2>

The Bottleneck block excels in several key areas:

- **<u>Compression Phase: Squeezing Features</u>**  
   - The first convolution in the Bottleneck block reduces the input channels.
   - This dimensionality reduction focuses on distilling the most critical information while discarding less important features.  

- **<u>Processing in the Bottleneck</u>**  
   - The reduced feature set undergoes transformations (convolutions and activations) to refine patterns efficiently.   
   - This step emphasizes core patterns while conserving computational resources.  

- **<u>Expansion Phase: Rebuilding Features</u>**  
   - The final convolution expands the channels back to ensure the network retains capacity for complex pattern modeling.
   - This combines the critical features from compression with the structural richness needed for downstream tasks.

- **<u>Promoting a Compact and Informative Representation</u>**  
   - By alternating between high and low-dimensional spaces, the Bottleneck prioritizes relevant features, retaining only the most useful information.  

---

**<u>Summary:</u>**  

The Bottleneck design balances **efficiency** by compressing features into a smaller channel depth, reducing computations and parameters, and **feature refinement** by emphasizing critical features, discarding redundancies, and retaining essential local relationships.


In [ ]:
class Bottleneck(nn.Module):
    '''Standard bottleneck'''
    
    def __init__(self, c1, c2, shortcut = True, g = 1, k = (3, 3), e = 0.5):
        '''Initialize a standard bottleneck with optional shortcut connection'''
        super().__init__()
        c_  = int(c2 * e) # hidden channels
        self.cv1 = Conv(c1, c_, k[0], 1) # first convolution layer
        self.cv2 = Conv(c_, c2, k[1], 1, g = g) # second convolution layer
        self.add = shortcut and c1 == c2 # whether to add input to output (shortcut connection)
    def forward(self, x):
        # forward propagation : input --> first convolution --> second convolution
        y = self.cv2(self.cv1(x))
        return x + y if self.add else y # add input to output if shortcut connection is enabled
    
class C2f(nn.Module):
    '''Implementation of the C2f module from YOLOv3.'''
    def __init__(self, c1, c2, n = 1, shortcut = False, g = 1, e = 0.5):
        '''Initialize a CSP bottleneck with 2 convolutions and n bottleneck layers '''
        super().__init__()
        self.c = int(c2 * e) # hidden channels
        self.cv1 = Conv(c1, 2 * self.c, 1, 1) # first convolution layer 
        self.cv2 = Conv((2 + n)*self.c, c2, 1) # second convolution layer
        self.m = nn.ModuleList(
            Bottleneck(self.c, self.c, shortcut, g, k = ((3, 3), (3, 3)), e = 1.0) for _ in range(n)
        ) # list of bottleneck layers
    
    def forward(self, x):
        # forward propagation : through C2f module
        y = list(self.cv1(x).chunk(2, 1)) # split output of first convolution into 2 parts
        y.extend(m(y[-1]) for m in self.m)
        
        return self.cv2(torch.cat(y, 1)) # concatenate outputs and pass through second convolution
    
    def forward_split(self, x):
        # forward propagation with split output (for inference optimization)
        y = list(self.cv1(x).split((self.c, self.c), 1)) # split output of first convolution into 2 parts
        y.extend(m(y[-1]) for m in self.m) # pass second part through bottleneck layers
        return self.cv2(torch.cat(y, 1)) # concatenate outputs and pass through second convolution
    
class C3k2(C2f):
    '''Faster implementation of C2f module with 2 convolutions'''
    def __init__(self, c1, c2, n = 1, c3k = False, e = 0.5, g = 1, shortcut = True):
        '''Initialize a C3k2 module , a faster CSP bottleneck with 2 convolutions and optional C3k block'''
        super().__init__(c1, c2, n, shortcut, g, e)
        self.m = nn.ModuleList(
            c3k(self.c, self.c,2, shortcut, g) if c3k else Bottleneck(self.c, self.c, shortcut, g) for _ in range(n)
        )
        
        

## <strong>C3k2 (T) Block</strong>

<h2><strong>Role of the C3k Block (when c3k = True)</strong></h2>

The C3k block excels in several key areas:

- **<u>Hierarchical Compression: Squeezing Features</u>**  
   - The initial convolutions reduce the input channels and partition features into smaller, more manageable subsets.  
   - This hierarchical compression retains the most critical information, optimizing for both efficiency and diversity of feature representation.  

- **<u>Multi-Stage Processing within C3k</u>**  
   - Each subset undergoes further refinement through a series of nested Bottleneck blocks.  
   - These blocks sequentially transform the compressed features to emphasize core patterns while discarding redundancies.  

- **<u>Final Expansion and Aggregation</u>**  
   - The outputs of the Bottleneck blocks are recombined and expanded through concatenation and the final convolution.  
   - This phase balances dimensionality and feature richness, ensuring the network is prepared for subsequent stages.  

- **<u>Promoting Feature Diversity and Refinement</u>**  
   - By incorporating multiple convolutional paths and iterative processing, the C3k block enhances the diversity of extracted patterns.  
   - This design ensures that both fine-grained and broader structural features are effectively captured.

---

**<u>Summary:</u>**  

The **C3k block** balances **multi-stage refinement** by leveraging nested Bottleneck blocks for hierarchical feature processing and **feature integration** by aggregating diverse patterns into a unified representation. This structure enhances the network's capacity to model complex patterns while maintaining computational efficiency.

In [20]:
class C2f(nn.Module):
    '''Faster implementation of CSP bottleneck with 2 convolutions'''
    
    def __init__(self, c1, c2, n = 1, shortcut = False, g = 1, e = 0.5):
        '''Initializes a CSP bottleneck with 2 convolutions and n bottleneck layers for faster processing'''
        
        super().__init__()
        self.c = int(c2 * e) # hidden channels
        self.cv1 = Conv(c1, 2 * self.c , 1, 1) # first convolution layer
        self.cv2 = Conv((2 + n)*self.c, c2, 1) # second convolution layer act = FReLU(c2) optional
        self.m = nn.ModuleList(
            Bottleneck(self.c, self.c, shortcut, g, k = ((3, 3), (3, 3)), e = 1.0) for _ in range(n)
        )
        
    def forward(self, x):
        # forward propagation : through c2f module
        y = list(self.cv1(x).chunk(2, 1)) # split output of first convolution into 2 parts
        y.extend(m(y[-1]) for m in self.m) # pass second part through bottleneck layers
        return self.cv2(torch.cat(y, 1)) # concatenate outputs and pass through second convolution

    def forward_split(self, x):
        # forward propagation with split output (for inference optimization)
        y = list(self.cv1(x).split((self.c, self.c), 1)) # split output of first convolution into 2 parts
        y.extend(m(y[-1]) for m in self.m) # pass second part through bottleneck layers
        return self.cv2(torch.cat(y, 1)) # concatenate outputs and pass through second convolution
    
class C3(nn.Module):
    '''CSP bottleneck with 3 convolutions'''
    
    def __init__(self, c1, c2, n = 1, shortcut = True, g = 1, e = 0.5):
        """Initializes a CSP bottleneck with 3 convolutions and n bottleneck layers for faster processing"""
        super().__init__()
        c_ = int(c2 * e) # hidden channels
        self.cv1 = Conv(c1, c_, 1, 1) # first convolution layer
        self.cv2 = Conv(c1, c_, 1, 1,) # second convolution layer
        self.cv3 = Conv(2*c_, c2, 1) # third convolution layer
        self.m = nn.Sequential(
            *(Bottleneck(c_, c_, shortcut, g, k = ((3, 3), (3, 3)), e = 1.0) for _ in range(n))
        )
        
    def forward(self, x):
        # forward propagation : through c3 module
        y1 = self.cv1(x) # first convolution on input
        y2 = self.cv2(x) # second convolution on input
        y1 = self.m(y1) # pass first convolution output through bottleneck layers
        return self.cv3(torch.cat((y1, y2), dim=1)) # concatenate outputs and pass through third convolution
    
class C3k(C3):
    '''C3k is a csp bottleneck module with customizable kernel size'''
    
    def __init__(self, c1, c2, n = 1, shortcut = True, g = 1, e = 0.5, k = 3):
        """Initializes a C3k module, a CSP bottleneck with customizable kernel size and n bottleneck layers for faster processing
        """
        super().__init__(c1, c2, n, shortcut, g, e)
        c_ = int(c2 * e) # hidden channels
        self.m = nn.Sequential(
            *(Bottleneck(c_, c_, shortcut, g, k = (k, k), e = 1.0) for _ in range(n))
        )
        
class C3k2(C2f):
    '''faster implementation of CSP bottleneck with 2 convolutions
    '''
    
    def __init__(self, c1, c2, n = 1, c3k = False, e = 0.5, g = 1, shortcut = True):
        """Initializes a C3k2 module, a faster CSP bottleneck with 2 convolutions and optional C3k block for faster processing
        """
        super().__init__(c1, c2, n, shortcut, g, e)
        self.m = nn.ModuleList(
            C3k(self.c, self.c,2, shortcut, g) if c3k else Bottleneck(self.c, self.c, shortcut, g) for _ in range(n)
        )
    

<h1><strong>SPPF BLOCK</strong></h1>

<h2><strong>Role of the SPPF Block (Spatial Pyramid Pooling – Fast)</strong></h2>

The SPPF block enhances object detection performance through:

- **<u>Multi-Scale Feature Aggregation:</u>**  
  Applies three max-pooling operations (kernel size = 5) to capture spatial information at multiple scales, combining fine details and broad context.

- **<u>Feature Fusion:</u>**  
  Concatenates outputs from pooling operations to create a rich, multi-scale feature map, enhancing the network’s ability to detect objects of varying sizes.

- **<u>Efficient Downsampling:</u>**  
  Preserves spatial relationships while reducing resolution, ensuring compact and meaningful feature representation.

- **<u>Optimized Design:</u>**  
  Streamlines traditional SPP, reducing computations while maintaining scalability for real-time applications.

This efficient fusion of multi-scale features boosts detection accuracy and speed.


In [21]:
class SPPF(nn.Module):
    '''Spatial pyramid pooling - fast with 5x less memory, compatible with ONNX Runtime'''
    
    def __init__(self, c1, c2, k = 5):
        
        '''
        Initializes the SPPF module with given input/output channels and kernel size
        This module is equivalent to SPP(k = (5, 9, 13)) but optimized for faster processing and reduced memory usage, making it compatible with ONNX Runtime.
        '''
        super().__init__()
        c_ = c1 // 2 # hidden channels
        self.cv1 = Conv(c1, c_, 1, 1) # first convolution layer
        self.cv2 = Conv(c_*4, c2, 1, 1) # second convolution layer
        self.m = nn.MaxPool2d(kernel_size = k, stride = 1, padding = k // 2) # max pooling layer with specified kernel size and padding for same output shape
        
    def forward(self, x):
        # forward propagation : through SPPF module
        y = [self.cv1(x)] # first convolution on input
        y.extend(self.m(y[-1]) for _ in range(3)) # apply max pooling three times to create spatial pyramid
        return self.cv2(torch.cat(y, 1)) # concatenate outputs and pass through second convolution
        